# 04 — Robustness checks and extensions

**Part 4.** This notebook stress-tests the stated-versus-revealed preference story and adds event-context and participant-segment analyses.

## Questions

1. Do other participants' ratings of the same date explain the focal participant's choice, or does their *own rating relative to peer consensus* still matter? (consensus / halo sensitivity)
2. Is yes propensity different in the limited-choice versus extensive-choice condition, and does it change over the sequence of dates? (choice architecture and order/fatigue)
3. How much do participants' stated preference weights differ from their retrospective account of what mattered during the event? (pre-event vs follow-up self-report)
4. Can participants be grouped by the size of the gap between stated weights and a simple person-level behavioral preference proxy?

## Read this before interpreting results

- Run `01_cleaning.ipynb` first. This notebook reads its `data/processed/clean_unscaled.csv` output; it does not depend on the repository's unprovided `src/` modules.
- In the data key, `condtn=1` means limited choice and `condtn=2` means extensive choice; `order` is the date number that night. `attr7_2` is a next-day retrospective report asking participants to recall which attributes mattered in their decisions. That is a change in *reported weights*, not proof of learning or a prospective preference update.
- The date ratings are subjective and collected after each interaction. The peer-rating sensitivity is therefore a consensus control, not an objective quality measure or a causal “halo effect” estimate.
- The person-level behavioral weights in the final section are an exploratory index from each participant's yes/no choices and post-date ratings. It is not the same estimator as the gender-level conditional logit in `02_core_model.ipynb` and should not be described as a stable individual preference or as causal.

**Marketing link:** these extensions connect preference measurement to recommendation quality, choice architecture, and audience segmentation. The data do not contain subscription prices, spending, retention, or lifetime revenue, so they cannot support CLV or pricing claims.

**Variable reference:** [Speed Dating Data Key](https://web.mit.edu/17.871/www/2009/speed_dating_codebook.pdf).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import chi2
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.discrete.conditional_models import ConditionalLogit
from IPython.display import display, Markdown

pd.set_option('display.max_columns', 30)
pd.set_option('display.width', 150)

ATTRS = ['attr', 'sinc', 'intel', 'fun', 'amb', 'shar']
PREF_PREFIXES = ['attr', 'sinc', 'intel', 'fun', 'amb', 'shar']
STATED = [f'{p}1_1' for p in PREF_PREFIXES]
FOLLOWUP = [f'{p}7_2' for p in PREF_PREFIXES]
SEED = 42
N_BOOT = 1000

# Set this to a path string if automatic discovery does not find the cleaned CSV.
DATA_PATH = None
if DATA_PATH is None:
    roots = [Path.cwd().resolve(), *Path.cwd().resolve().parents]
    candidates = []
    for root in roots:
        candidates.extend([
            root / 'data' / 'processed' / 'clean_unscaled.csv',
            root / 'processed' / 'clean_unscaled.csv',
        ])
    DATA_PATH = next((p for p in candidates if p.is_file()), None)

if DATA_PATH is None:
    raise FileNotFoundError(
        'Could not find data/processed/clean_unscaled.csv. Run 01_cleaning.ipynb, '
        'or set DATA_PATH above to the cleaned CSV location.'
    )

DATA_PATH = Path(DATA_PATH)
data = pd.read_csv(DATA_PATH)
required = ['iid', 'pid', 'wave', 'gender', 'dec', *ATTRS]
missing = [c for c in required if c not in data.columns]
if missing:
    raise ValueError(f'The cleaned CSV is missing required columns: {missing}')

for col in ['iid', 'pid', 'wave', 'gender', 'dec', 'condtn', 'order', 'match', *ATTRS, *STATED, *FOLLOWUP]:
    if col in data.columns:
        data[col] = pd.to_numeric(data[col], errors='coerce')
data = data.loc[data['dec'].isin([0, 1])].copy()

print(f'Source: {DATA_PATH}')
print(f'{len(data):,} date rows; {data.iid.nunique():,} participants; {data.wave.nunique():,} waves')
print(f'Observed yes rate: {data.dec.mean():.3f}')

## 1. Sample support and data coverage

First check that the fields needed by each extension are present and that condition is constant within participant, as expected for a between-person event condition. The wave-by-condition table tells us whether we can compare condition groups *within the same wave*; without overlap, condition is confounded with event wave in this dataset.

In [ ]:
coverage = pd.DataFrame({
    'field': ['condtn', 'order', 'match', *STATED, *FOLLOWUP],
    'available': [c in data.columns for c in ['condtn', 'order', 'match', *STATED, *FOLLOWUP]],
})
display(coverage)

if 'condtn' in data.columns:
    condtn_per_person = data.groupby('iid')['condtn'].nunique(dropna=True)
    print(f"Participants with >1 recorded condition value: {(condtn_per_person > 1).sum()}")
    person_condition = data.groupby('iid', as_index=False).agg(
        gender=('gender', 'first'), wave=('wave', 'first'), condtn=('condtn', 'first'),
        n_dates=('dec', 'size'), yes_rate=('dec', 'mean'),
        **({'mutual_match_rate': ('match', 'mean')} if 'match' in data.columns else {})
    )
    display(person_condition.groupby(['condtn', 'gender'], dropna=False).agg(
        participants=('iid', 'nunique'), median_dates=('n_dates', 'median'),
        mean_yes_rate=('yes_rate', 'mean'),
        **({'mean_mutual_match_rate': ('mutual_match_rate', 'mean')} if 'match' in person_condition.columns else {})
    ).round(3))

    condition_by_wave = pd.crosstab(person_condition['wave'], person_condition['condtn'])
    display(condition_by_wave)
    condition_levels = set(person_condition['condtn'].dropna().unique())
    overlap_waves = condition_by_wave.index[condition_by_wave.reindex(columns=sorted(condition_levels), fill_value=0).gt(0).all(axis=1)].tolist() if len(condition_levels) > 1 else []
    print(f'Waves containing both condition levels: {overlap_waves}')

## 2. Consensus / halo sensitivity

For each date row `(iid, pid, wave)`, construct the mean rating that **other participants in the same wave** gave the focal participant's date partner. The focal row is left out of that average. This direction matters: we aggregate rows where the partner is the rated person; the dataset's `attr_o`-style fields record the opposite direction.

We then decompose the focal rating into (a) peer consensus and (b) the focal person's deviation from that consensus. Both are put on the same within-gender scale. The baseline is the core-style conditional logit on the focal person's six ratings. The augmented model lets consensus and the person's own deviation have separate coefficients. A likelihood-ratio test asks whether the split adds explanatory information. A persistent deviation coefficient suggests choices contain idiosyncratic signal beyond consensus; it does not prove that consensus creates a halo.

In [ ]:
def informative_rows(frame, predictors):
    d = frame.dropna(subset=['iid', 'dec', *predictors]).copy()
    informative_iids = d.groupby('iid')['dec'].nunique()
    informative_iids = informative_iids[informative_iids.eq(2)].index
    return d[d['iid'].isin(informative_iids)].copy()


def fit_conditional_logit(frame, predictors):
    d = informative_rows(frame, predictors)
    if d['iid'].nunique() < 2:
        raise ValueError('Too few participants with both yes and no outcomes for conditional logit.')
    X = d[predictors].astype(float)
    result = ConditionalLogit(d['dec'].astype(int), X, groups=d['iid']).fit(disp=False, maxiter=300)
    return result, d

if 'pid' not in data.columns or 'wave' not in data.columns:
    raise ValueError('The consensus analysis requires pid and wave.')

halo = data.dropna(subset=['iid', 'pid', 'wave', 'gender', 'dec']).copy()
for attr in ATTRS:
    keys = halo.groupby(['wave', 'pid'])[attr]
    total = keys.transform('sum')
    n_raters = keys.transform('count')
    focal_has_rating = halo[attr].notna().astype(int)
    n_other_raters = n_raters - focal_has_rating
    peer_total = total - halo[attr].fillna(0)
    halo[f'peer_{attr}'] = (peer_total / n_other_raters.where(n_other_raters > 0)).where(n_other_raters > 0)
    halo[f'deviation_{attr}'] = halo[attr] - halo[f'peer_{attr}']

peer_cols = [f'peer_{a}' for a in ATTRS]
dev_cols = [f'deviation_{a}' for a in ATTRS]
halo = halo.dropna(subset=[*ATTRS, *peer_cols]).copy()
halo = informative_rows(halo, [*ATTRS, *peer_cols])

halo_rows = []
for gender_value, gender_label in [(0, 'Women'), (1, 'Men')]:
    g = halo.loc[halo['gender'].eq(gender_value)].copy()
    if g.empty:
        continue
    own_z, peer_z, dev_z = [], [], []
    for a in ATTRS:
        mean = g[a].mean()
        sd = g[a].std(ddof=0)
        if not np.isfinite(sd) or sd == 0:
            raise ValueError(f'Cannot standardize {a} ratings for {gender_label}.')
        own_name, peer_name, dev_name = f'own_{a}_z', f'peer_{a}_z', f'dev_{a}_z'
        g[own_name] = (g[a] - mean) / sd
        g[peer_name] = (g[f'peer_{a}'] - mean) / sd
        # Same centering/scaling as the own rating, so own_z == peer_z + deviation_z.
        g[dev_name] = g[own_name] - g[peer_name]
        own_z.append(own_name)
        peer_z.append(peer_name)
        dev_z.append(dev_name)

    baseline, db = fit_conditional_logit(g, own_z)
    split, ds = fit_conditional_logit(g, [*peer_z, *dev_z])
    lr = max(0.0, 2 * (split.llf - baseline.llf))
    df_lr = len(split.params) - len(baseline.params)
    p_lr = chi2.sf(lr, df_lr) if df_lr > 0 else np.nan
    halo_rows.append({
        'gender': gender_label, 'n_dates': len(ds), 'n_people': ds.iid.nunique(),
        'baseline_llf': baseline.llf, 'split_model_llf': split.llf,
        'LR_stat': lr, 'LR_df': df_lr, 'LR_p': p_lr,
    })
    for a, own_name, peer_name, dev_name in zip(ATTRS, own_z, peer_z, dev_z):
        halo_rows.append({
            'gender': gender_label, 'attribute': a,
            'baseline_own_beta': baseline.params[own_name],
            'peer_consensus_beta': split.params[peer_name],
            'own_minus_peer_beta': split.params[dev_name],
        })

display(pd.DataFrame(halo_rows).round(4))

### How to read this check

- `peer_consensus_beta` is the association between other daters' average rating of the partner and the participant's yes decision, conditional on the participant's own rating.
- `own_minus_peer_beta` captures the participant's rating relative to the other daters' average, conditional on consensus.
- The likelihood-ratio comparison is a model diagnostic. People rate multiple dates and partners are rated by multiple people, so the conditional-logit standard errors do not account for every cross-person dependency. Treat p-values as exploratory and emphasize coefficient direction and size.

## 3. Choice condition and choice overload

`condtn` is a between-participant event condition, so it cannot be identified in a participant fixed-effects model. We estimate an exploratory row-level logistic model with standard errors clustered by participant. If both conditions occur within the same waves, the primary comparison uses only those overlap waves and includes wave fixed effects. If there is no within-wave overlap, the notebook reports a gender-adjusted association without wave fixed effects and flags the condition/wave confounding.

The condition contrast is not automatically causal: the codebook labels limited versus extensive choice, but the data do not guarantee that condition assignment is independent of wave or every other event difference.

In [ ]:
if 'condtn' not in data.columns:
    print('condtn is absent; skip condition model.')
else:
    cond = data.dropna(subset=['iid', 'dec', 'gender', 'condtn', 'wave']).copy()
    cond['condtn'] = cond['condtn'].astype(int)
    levels = sorted(cond['condtn'].unique())
    print(f'Condition values present: {levels} (data key: 1=limited, 2=extensive)')

    if len(levels) == 2 and overlap_waves:
        cond_fit = cond.loc[cond['wave'].isin(overlap_waves)].copy()
        condition_formula = 'dec ~ C(condtn) + C(gender) + C(wave)'
        condition_note = 'Wave-adjusted estimate on waves with both conditions.'
    else:
        cond_fit = cond
        condition_formula = 'dec ~ C(condtn) + C(gender)'
        condition_note = 'No within-wave condition comparison available; condition is potentially confounded with wave.'

    condition_result = smf.glm(
        formula=condition_formula,
        data=cond_fit,
        family=sm.families.Binomial()
    ).fit(cov_type='cluster', cov_kwds={'groups': cond_fit['iid']})
    print(condition_note)
    print(f'Formula: {condition_formula}; rows={len(cond_fit):,}; participants={cond_fit.iid.nunique():,}')

    condition_terms = [term for term in condition_result.params.index if term.startswith('C(condtn)')]
    if condition_terms:
        term = condition_terms[-1]
        beta = condition_result.params[term]
        ci_low, ci_high = condition_result.conf_int().loc[term]
        condition_result_table = pd.DataFrame([{
            'contrast': 'extensive vs limited (reference condition 1)',
            'log_odds': beta, 'odds_ratio': np.exp(beta),
            'OR_95CI_low': np.exp(ci_low), 'OR_95CI_high': np.exp(ci_high),
            'p_clustered_by_iid': condition_result.pvalues[term],
        }])
        display(condition_result_table.round(4))
    else:
        print('Could not estimate a condition contrast; check whether two condition values are present.')

## 4. Order and possible fatigue

`order` is the sequence position of each date that night. To make effect size comparable across participants with different numbers of dates, rescale each person's observed range to 0 (first) through 1 (last). Fit conditional logits with participant fixed effects, first with the six rating controls, then with a linear order term, then with a quadratic term. This asks whether yes propensity changes over the event and whether a straight line is adequate.

An order association is not necessarily fatigue: date quality may vary by order, and the study's ordering process and event context should be considered when interpreting it.

In [ ]:
if 'order' not in data.columns:
    print('order is absent; skip order analysis.')
else:
    order_data = data.dropna(subset=['iid', 'gender', 'dec', 'order', *ATTRS]).copy()
    order_min = order_data.groupby('iid')['order'].transform('min')
    order_max = order_data.groupby('iid')['order'].transform('max')
    order_data['order_rel'] = (order_data['order'] - order_min) / (order_max - order_min).replace(0, np.nan)
    order_data['order_rel_sq'] = order_data['order_rel'] ** 2
    order_data = order_data.dropna(subset=['order_rel', 'order_rel_sq']).copy()
    order_data = informative_rows(order_data, [*ATTRS, 'order_rel', 'order_rel_sq'])

    order_rows = []
    for gender_value, gender_label in [(0, 'Women'), (1, 'Men')]:
        g = order_data.loc[order_data['gender'].eq(gender_value)].copy()
        if g.empty:
            continue
        rating_z = []
        for a in ATTRS:
            sd = g[a].std(ddof=0)
            if not np.isfinite(sd) or sd == 0:
                continue
            z = f'{a}_z'
            g[z] = (g[a] - g[a].mean()) / sd
            rating_z.append(z)
        if len(rating_z) != len(ATTRS):
            print(f'Skipping {gender_label}: at least one rating has zero variance.')
            continue

        base, d_base = fit_conditional_logit(g, rating_z)
        linear, d_linear = fit_conditional_logit(g, [*rating_z, 'order_rel'])
        quadratic, d_quad = fit_conditional_logit(g, [*rating_z, 'order_rel', 'order_rel_sq'])

        lr_order = max(0.0, 2 * (linear.llf - base.llf))
        p_order = chi2.sf(lr_order, 1)
        lr_curve = max(0.0, 2 * (quadratic.llf - linear.llf))
        p_curve = chi2.sf(lr_curve, 1)
        b_linear = linear.params['order_rel']
        b1 = quadratic.params['order_rel']
        b2 = quadratic.params['order_rel_sq']
        order_rows.append({
            'gender': gender_label, 'n_dates': len(d_quad), 'n_people': d_quad.iid.nunique(),
            'linear_order_beta': b_linear, 'linear_OR_first_to_last': np.exp(b_linear),
            'LR_add_order': lr_order, 'p_add_order': p_order,
            'quadratic_beta': b2, 'quadratic_OR_first_to_last': np.exp(b1 + b2),
            'LR_add_curvature': lr_curve, 'p_add_curvature': p_curve,
        })
    display(pd.DataFrame(order_rows).round(4))

### Interpretation guide

- `linear_OR_first_to_last < 1` means lower yes odds toward the last observed date in the linear specification.
- `p_add_curvature` tests whether the quadratic term improves fit beyond the linear order effect. It is a shape check, not a test of psychological fatigue by itself.
- These are conditional-logit likelihood comparisons; the same participant is the unit of conditioning, but repeated partner/event dependence is not fully modeled.

## 5. Pre-event stated preferences versus next-day recollection

The follow-up `attr7_2` family asks participants to retrospectively allocate importance to the attributes that mattered in their event decisions. Compare it with the initial `*1_1` preference allocation using a paired, participant-level bootstrap. The comparison is compositional: all six weights are normalized to sum to 100 before deltas are calculated.

A shift means the reported importance changed between sign-up and recall. It may reflect learning, memory, justification, measurement noise, or the change in question wording; the data cannot distinguish these mechanisms.

In [ ]:
pref_cols = [c for c in [*STATED, *FOLLOWUP] if c in data.columns]
if not all(c in data.columns for c in [*STATED, *FOLLOWUP]):
    print('One or more preference columns are absent; skip paired preference comparison.')
else:
    repeated_consistency = data.groupby('iid')[[*STATED, *FOLLOWUP]].nunique(dropna=True).gt(1).any(axis=1)
    print(f'Participants with inconsistent repeated preference values across date rows: {repeated_consistency.sum()}')

    people = data.groupby('iid')[[*STATED, *FOLLOWUP, 'gender', 'wave']].first()
    pre_raw = people[STATED].apply(pd.to_numeric, errors='coerce')
    post_raw = people[FOLLOWUP].apply(pd.to_numeric, errors='coerce')

    def normalize_composition(values):
        values = values.where(values.ge(0))
        totals = values.sum(axis=1, min_count=values.shape[1])
        return values.div(totals.where(totals > 0), axis=0) * 100

    pre = normalize_composition(pre_raw)
    post = normalize_composition(post_raw)
    pre.columns = ATTRS
    post.columns = ATTRS
    complete_pair = pre.notna().all(axis=1) & post.notna().all(axis=1)
    pre_pair, post_pair = pre.loc[complete_pair], post.loc[complete_pair]
    delta = post_pair - pre_pair
    n_pair = len(delta)
    print(f'Complete paired participants: {n_pair:,} / {len(people):,}')

    if n_pair:
        rng = np.random.default_rng(SEED)
        delta_values = delta.to_numpy()
        boot_means = np.empty((N_BOOT, len(ATTRS)))
        for b in range(N_BOOT):
            idx = rng.integers(0, n_pair, size=n_pair)
            boot_means[b] = delta_values[idx].mean(axis=0)
        ci = np.quantile(boot_means, [0.025, 0.975], axis=0)
        paired_results = pd.DataFrame({
            'attribute': ATTRS,
            'pre_mean': pre_pair.mean().to_numpy(),
            'followup_mean': post_pair.mean().to_numpy(),
            'mean_change_pp': delta.mean().to_numpy(),
            'bootstrap_95CI_low': ci[0], 'bootstrap_95CI_high': ci[1],
            'share_increased': (delta > 0).mean().to_numpy(),
        })
        display(paired_results.round(2))
        total_variation = 0.5 * delta.abs().sum(axis=1)
        print(f'Mean within-person pre/follow-up reallocation: {total_variation.mean():.1f} points; '
              f'median={total_variation.median():.1f}; 95th percentile={total_variation.quantile(.95):.1f}')

## 6. Segment participants by the size of their say–do gap

For each participant with enough yes and no decisions, calculate the within-person standardized rating difference between dates they accepted and rejected for each attribute. Positive differences are converted into a six-attribute behavioral importance proxy that sums to 100. Compare that proxy with the participant's normalized pre-event `*1_1` stated weights using total-variation distance:

`gap = 0.5 × sum(abs(stated_weight − behavioral_weight))`

The result ranges from 0 to 100 percentage points. A high gap means more of the 100-point allocation would need to move to make the two profiles agree. Terciles are descriptive segments; they are not discovered latent classes. Because the behavioral proxy is calculated from the same event's decisions and date ratings, do not claim it predicts future matches without a held-out validation sample.

In [ ]:
def derive_gap_segments(min_each=2):
    required_segment = ['iid', 'dec', *ATTRS, *STATED]
    d = data.dropna(subset=required_segment).copy()
    for a in ATTRS:
        means = d.groupby('iid')[a].transform('mean')
        sds = d.groupby('iid')[a].transform(lambda s: s.std(ddof=0))
        d[f'z_{a}'] = ((d[a] - means) / sds.replace(0, np.nan)).fillna(0.0)

    out = []
    no_positive_signal = 0
    too_few_choices = 0
    for iid, g in d.groupby('iid', sort=False):
        yes = g.loc[g['dec'].eq(1)]
        no = g.loc[g['dec'].eq(0)]
        if len(yes) < min_each or len(no) < min_each:
            too_few_choices += 1
            continue
        differences = np.array([yes[f'z_{a}'].mean() - no[f'z_{a}'].mean() for a in ATTRS], dtype=float)
        positive = np.clip(differences, 0, None)
        if not np.isfinite(positive).all() or positive.sum() <= 0:
            no_positive_signal += 1
            continue
        behavior = positive / positive.sum() * 100
        stated = pd.to_numeric(g.iloc[0][STATED], errors='coerce').to_numpy(dtype=float)
        if not np.isfinite(stated).all() or stated.sum() <= 0:
            continue
        stated = stated / stated.sum() * 100
        gap = 0.5 * np.abs(stated - behavior).sum()
        row = {
            'iid': iid, 'gender': g['gender'].iloc[0] if 'gender' in g else np.nan,
            'wave': g['wave'].iloc[0] if 'wave' in g else np.nan,
            'n_dates': len(g), 'n_yes': len(yes), 'n_no': len(no),
            'say_do_gap_pp': gap,
            'top_stated_attribute': ATTRS[int(np.argmax(stated))],
            'top_behavior_attribute': ATTRS[int(np.argmax(behavior))],
        }
        if 'match' in g.columns:
            row['mutual_match_rate'] = g['match'].mean()
        for a, value in zip(ATTRS, stated):
            row[f'stated_{a}'] = value
        for a, value in zip(ATTRS, behavior):
            row[f'behavior_{a}'] = value
        out.append(row)

    result = pd.DataFrame(out)
    result.attrs['too_few_choices'] = too_few_choices
    result.attrs['no_positive_signal'] = no_positive_signal
    if result.empty:
        return result
    rank_pct = result['say_do_gap_pp'].rank(method='average', pct=True)
    result['gap_segment'] = pd.cut(
        rank_pct, bins=[0, 1/3, 2/3, 1.0000001],
        labels=['Low gap', 'Medium gap', 'High gap'], include_lowest=True
    )
    return result

segments_2 = derive_gap_segments(min_each=2)
print(f"Participants eligible (>=2 yes and >=2 no): {len(segments_2):,}; "
      f"excluded for too few choices: {segments_2.attrs.get('too_few_choices', 0):,}; "
      f"excluded because no positive behavioral signal: {segments_2.attrs.get('no_positive_signal', 0):,}")
if not segments_2.empty:
    agg = {
        'participants': ('iid', 'nunique'),
        'mean_gap_pp': ('say_do_gap_pp', 'mean'),
        'median_gap_pp': ('say_do_gap_pp', 'median'),
        'top_behavior_attribute': ('top_behavior_attribute', lambda s: s.value_counts().index[0]),
    }
    if 'mutual_match_rate' in segments_2.columns:
        agg['mean_mutual_match_rate'] = ('mutual_match_rate', 'mean')
    segment_summary = segments_2.groupby('gap_segment', observed=False).agg(**agg)
    display(segment_summary.round(3))

    profile_cols = [f'stated_{a}' for a in ATTRS] + [f'behavior_{a}' for a in ATTRS]
    segment_profiles = segments_2.groupby('gap_segment', observed=False)[profile_cols].mean()
    display(segment_profiles.round(1))

# Sensitivity to a stricter minimum number of yes/no observations for each person.
segments_3 = derive_gap_segments(min_each=3)
sensitivity = pd.DataFrame([
    {'minimum_each_outcome': 2, 'participants': len(segments_2),
     'median_gap_pp': segments_2['say_do_gap_pp'].median() if len(segments_2) else np.nan},
    {'minimum_each_outcome': 3, 'participants': len(segments_3),
     'median_gap_pp': segments_3['say_do_gap_pp'].median() if len(segments_3) else np.nan},
])
display(sensitivity.round(2))

## 7. What to report

Use the tables above to answer the following without overstating the evidence:

1. **Consensus sensitivity:** Do consensus ratings add fit? For which attributes does the participant's own rating relative to consensus retain an association with yes decisions?
2. **Choice architecture:** Is the extensive-choice condition associated with a different yes rate? Is the contrast still estimable within waves that contain both conditions? If not, describe it as an event-level association, not a causal choice-overload effect.
3. **Order:** Does yes propensity rise or fall from first to last date? Does the quadratic term improve fit? Call this an order effect unless fatigue is separately identified.
4. **Retrospective preference:** Which attributes show the largest average pre-event to follow-up shifts, and how large is the typical total reallocation? Describe this as recall/reported-weight change.
5. **Segments:** What do the high-gap participants look like in their stated and behavior-linked profiles? Are mutual-match rates different descriptively? This is a useful targeting hypothesis for a future platform test, not proof that a segment-specific recommendation policy will improve matches.

### Marketing-analytics interpretation

The strongest marketing framing is **preference-based recommendation and choice architecture**: stated preferences, event behavior, and partner consensus may each add different signals to matching. The gap segments suggest a testable onboarding strategy (for example, ask users to allocate preferences, then compare their top recommendations with observed interaction feedback). This dataset does not contain real app recommendations, exposure logs, subscription behavior, retention, or revenue, so it cannot estimate a commercial lift, CLV, or an optimal price plan.

### Method references

- [Speed Dating Data Key](https://web.mit.edu/17.871/www/2009/speed_dating_codebook.pdf) — field definitions and questionnaire timing.
- [Fisman, Iyengar, Kamenica & Simonson (2006)](https://faculty.chicagobooth.edu/emir.kamenica/documents/genderDifferences.pdf) — original study context.
- The core-model notebook uses conditional fixed-effects logit and participant-level bootstrap; this notebook follows its participant grouping for conditional logits but treats cross-partner and event dependence as an important limitation.